# Example 01: Creating an Agent
To analyze your own agent you must implement the interface `IAgent`.

Agents get called everytime it is their turn to move with the current state of the board. 

The framework also handles rolling the dice for you.

To make implementation easier, it also provides a complete list of all legal actions the agent may execute in the current position.

Implement the `choose_action(self, state, actions)` method to return the action of your choice. 

`IAgent` provides a field `self.transition_model`, which may be used to figure out the state of the board after a certain action is taken. See below for usage example.


In [18]:
# Interface your agent must implement
from bg_agents.iagent import IAgent

# Common types used throughout the project
from bg_game.game_types import (
    # Indices of fields: (from, to) 
    Move,
    # May contain 1 to 4 individual moves
    Action, 
    # Board from your agents perspective
    AgentPerspectiveState, 
)

In [19]:
class SimpleAgent(IAgent):

    def choose_action(
            self, 
            state: AgentPerspectiveState, 
            actions: list[Action] 
        ) -> Action:
        """
        Always returns the first legal action,
        i.e. essentially random.
        """
        return actions[0]
        

The framework guarantees that `actions` is never empty. 

Incase the agent is blocked (no legal moves), the framework will not call the agent, but skip his turn. 

Therefore our `SimpleAgent` is guaranteed to work; but clearly he is not very smart.

We can do better by predicting the outcome of each action, and choosing the one with the best outcome:

In [20]:
class SmartAgent(IAgent):
    
    def choose_action(
            self, 
            state: AgentPerspectiveState, 
            actions: list[Action]
        ):
        """
        Simple implementation of a utility based agent,
        with one-step lookahead.
        """

        # Compute the utility of each action's outcome using self.transition_model
        action_utilities = [
            (action, self._utility(self.transition_model.result(state, action)))
            for action in actions
        ]
        
        # Find the action that leads to the highest utility
        best_action, _ = max(action_utilities, key=lambda x: x[1])
        return best_action
    
    def _utility(self, state: AgentPerspectiveState):
        """
        Calculates the utility of a certain state. 
        The higher the utility, the more desirable the state.
        """
        utility = 0

        # Increase utility for hit enemies
        #utility += state.bar_enemy

        # Increase utility for off moves
        #utility += 2 * state.off_me

        # Enemy checkers are signed, own checkers are unsigned
        for amount_checkers in state.points:
            # This agent likes to play it safe
            if amount_checkers >= 2: 
                utility += 1

        return utility
        

Our `SmartAgent` now has a clear strategy: avoid blops, hit enemy checkers and move own checkers off the board. 

But is he really any better than our `SimpleAgent` from earlier?

Let's find out by running a simulation:

In [21]:
# Lab orchestrates the simulations
from bg_lab.lab import Lab

# Inside the lab, each agent is assigned a color
from bg_game.game_types import (
    Color, WHITE, BLACK
)

# The lab produces a DataFrame
import pandas

In [22]:
# Simple VS Simple
lab = Lab()

agents = {WHITE: SimpleAgent(), BLACK: SimpleAgent()}

df = lab.compare_agents(
    white_agent=agents[WHITE],
    black_agent=agents[BLACK],
    n_matches=100
    )

winrate = (df['winner'] == WHITE).mean()
winrate

np.float64(0.49)

Unsurprisingly, the winrate is around 50%. 

Let's see how the `SmartAgent` performs:

In [23]:
# Smart VS Simple
lab = Lab()

agents = {WHITE: SmartAgent(), BLACK: SimpleAgent()}

df = lab.compare_agents(
    white_agent=agents[WHITE],
    black_agent=agents[BLACK],
    n_matches=100
    )

winrate  = (df['winner'] == WHITE).mean()
winrate

np.float64(0.81)

This project also provides some agents you can play against, such as the `RandomAgent` (which randomly chooses an action) or the `UtilityBasedAgent` (which works similiar to our `SmartAgent`).

In [24]:
from bg_agents.simple_utility_based_agent import SimpleUtilityBasedAgent

In [25]:
# Smart VS SimpleUtilityBasedAgent
lab = Lab()

agents = {WHITE: SmartAgent(), BLACK: SimpleUtilityBasedAgent()}

df = lab.compare_agents(
    white_agent=agents[WHITE],
    black_agent=agents[BLACK],
    n_matches=300
    )

winrate  = (df['winner'] == WHITE).mean()
winrate

np.float64(0.33666666666666667)

The dataframe provides many other interesting metrics, and you can even add your own metric. 

This will be explained in the next notebooks.